# 🚲 Bike-Sharing Demand Prediction — Data Visualization Project

### A Major Project on Exploratory Data Analysis (EDA) & Data Visualization

---

**Author:** _Your Name Here_
**Course/Subject:** Data Science / Machine Learning
**Dataset:** Seoul Bike Sharing Demand Dataset (8,760 hourly records)


## 1. Problem Statement

Bike-Sharing demand is predicting the number of bikes that will be rented from a
Bike-Sharing system at a given time based on **weather**, **day of the week**, and
**time of day**.

The purpose of this project is to build a predictive model to accurately forecast
bike rental demand in order to:
- Optimize bike allocation across stations
- Improve the overall efficiency of the bike-sharing system
- Reduce shortages during peak demand and idle bikes during low demand

Before building any predictive model, this notebook focuses on **understanding the
data deeply through visualization** — uncovering patterns, trends, and relationships
between rental demand and the various weather/time factors.


## 2. Objectives of this Notebook

1. Load and clean the Bike-Sharing dataset
2. Perform data understanding (structure, types, missing values, statistics)
3. Engineer useful time-based features (Day of Week, Month, Weekend flag)
4. Visualize demand patterns across:
   - Time of day (Hour)
   - Day of the week
   - Seasons & Months
   - Weather conditions (Temperature, Humidity, Wind, Rainfall, Snowfall, Visibility, Solar Radiation)
   - Holidays & Functioning Days
5. Study correlations between numerical features and bike demand
6. Summarize key business insights derived from the visualizations


## 3. Setup — Import Libraries

Run this cell first. All libraries used below (`pandas`, `numpy`, `matplotlib`,
`seaborn`, `plotly`) are pre-installed in Google Colab.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Display & style settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully')


## 4. Load the Dataset

**Running in Google Colab:** upload `Bike_Sharing_Demand.csv` using the file upload
cell below (or mount your Google Drive and update the path).


In [ ]:
# ---- OPTION A: Upload directly in Colab ----
from google.colab import files
uploaded = files.upload()   # Select 'Bike_Sharing_Demand.csv' when prompted
filename = list(uploaded.keys())[0]

# ---- OPTION B: If using Google Drive instead, comment Option A above and use:
# from google.colab import drive
# drive.mount('/content/drive')
# filename = '/content/drive/MyDrive/Bike_Sharing_Demand.csv'

# The file has special characters (°) so we read it with the correct encoding
df = pd.read_csv(filename, encoding='cp1252')
print('Shape of dataset:', df.shape)
df.head()


## 5. Data Understanding

Let's inspect the structure, data types, and check for missing/duplicate values.


In [ ]:
df.info()


In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())


In [ ]:
df.describe().T


**Column Reference:**

| Column | Description |
|---|---|
| Date | Date of record (dd/mm/yyyy) |
| Rented Bike Count | Target variable — number of bikes rented in that hour |
| Hour | Hour of the day (0–23) |
| Temperature(°C) | Air temperature |
| Humidity(%) | Relative humidity |
| Wind speed (m/s) | Wind speed |
| Visibility (10m) | Visibility distance |
| Dew point temperature(°C) | Dew point |
| Solar Radiation (MJ/m2) | Solar radiation |
| Rainfall(mm) | Rainfall amount |
| Snowfall (cm) | Snowfall amount |
| Seasons | Winter / Spring / Summer / Autumn |
| Holiday | Holiday / No Holiday |
| Functioning Day | Whether the bike-sharing system was operational (Yes/No) |


## 6. Data Cleaning & Feature Engineering

We convert `Date` to a proper datetime object and derive new time-based features
that are essential for the "Day of the week" and "time of day" aspects mentioned
in the problem statement.


In [ ]:
# Rename columns for easier handling
df.columns = ['Date', 'Rented_Bike_Count', 'Hour', 'Temperature', 'Humidity',
              'Wind_Speed', 'Visibility', 'Dew_Point_Temp', 'Solar_Radiation',
              'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning_Day']

# Convert Date column
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

# Feature engineering
df['Month'] = df['Date'].dt.month_name()
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Is_Weekend'] = df['Date'].dt.dayofweek.isin([5, 6]).map({True: 'Weekend', False: 'Weekday'})

# Keep only functioning days for demand analysis (system was actually renting bikes)
df_active = df[df['Functioning_Day'] == 'Yes'].copy()

print('Full dataset shape:', df.shape)
print('Active (Functioning Day = Yes) dataset shape:', df_active.shape)
df.head()


## 7. Distribution of the Target Variable — Rented Bike Count

Understanding the shape of demand (skewness, outliers) is the first step before
any modelling.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df_active['Rented_Bike_Count'], bins=40, kde=True, color='#2E86AB', ax=axes[0])
axes[0].set_title('Distribution of Rented Bike Count')
axes[0].set_xlabel('Rented Bike Count')

sns.boxplot(x=df_active['Rented_Bike_Count'], color='#F18F01', ax=axes[1])
axes[1].set_title('Boxplot of Rented Bike Count (Outlier Check)')

plt.tight_layout()
plt.show()

print(f"Average hourly demand: {df_active['Rented_Bike_Count'].mean():.0f} bikes")
print(f"Median hourly demand:  {df_active['Rented_Bike_Count'].median():.0f} bikes")
print(f"Max hourly demand:     {df_active['Rented_Bike_Count'].max()} bikes")


**Insight:** The demand distribution is right-skewed — most hours see moderate demand, with fewer hours of very high demand (likely rush hours).

## 8. Demand by Time of Day (Hour)

This directly addresses the *"time of day"* factor from the problem statement.


In [ ]:
hourly_avg = df_active.groupby('Hour')['Rented_Bike_Count'].mean().reset_index()

fig = px.line(hourly_avg, x='Hour', y='Rented_Bike_Count', markers=True,
              title='Average Bike Rentals by Hour of the Day',
              labels={'Rented_Bike_Count': 'Avg. Rented Bike Count', 'Hour': 'Hour of Day'})
fig.update_traces(line_color='#2E86AB', line_width=3)
fig.update_layout(template='plotly_white', xaxis=dict(dtick=1))
fig.show()


**Insight:** Demand peaks sharply around **8 AM** and **6 PM**, matching typical office/school commute hours, with a smaller midday bump and lowest demand in the early morning hours (2–5 AM).

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_active, x='Hour', y='Rented_Bike_Count', palette='coolwarm')
plt.title('Hourly Distribution & Variability of Bike Rentals')
plt.xlabel('Hour of Day')
plt.ylabel('Rented Bike Count')
plt.show()


## 9. Demand by Day of the Week

This directly addresses the *"day of the week"* factor from the problem statement.


In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_avg = df_active.groupby('Day_of_Week')['Rented_Bike_Count'].mean().reindex(day_order).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=day_avg, x='Day_of_Week', y='Rented_Bike_Count', palette='viridis')
plt.title('Average Bike Rentals by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Avg. Rented Bike Count')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(data=df_active, x='Hour', y='Rented_Bike_Count', hue='Is_Weekend',
             estimator='mean', ci=None, palette={'Weekday': '#2E86AB', 'Weekend': '#F18F01'}, linewidth=3)
plt.title('Hourly Demand Pattern: Weekday vs Weekend')
plt.xlabel('Hour of Day')
plt.ylabel('Avg. Rented Bike Count')
plt.xticks(range(0, 24))
plt.legend(title='Day Type')
plt.show()


**Insight:** Weekdays show the classic **two-peak commute pattern** (morning + evening rush), while weekends show a **single broader midday peak**, suggesting weekend usage is more recreational than commute-driven.

## 10. Demand by Season & Month


In [ ]:
season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
season_avg = df_active.groupby('Seasons')['Rented_Bike_Count'].mean().reindex(season_order).reset_index()

fig = px.bar(season_avg, x='Seasons', y='Rented_Bike_Count', color='Seasons',
             title='Average Bike Rentals by Season',
             labels={'Rented_Bike_Count': 'Avg. Rented Bike Count'},
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()


In [ ]:
month_order = ['January','February','March','April','May','June','July',
               'August','September','October','November','December']
month_avg = df_active.groupby('Month')['Rented_Bike_Count'].mean().reindex(month_order).dropna().reset_index()

plt.figure(figsize=(14, 6))
sns.lineplot(data=month_avg, x='Month', y='Rented_Bike_Count', marker='o',
             color='#C73E1D', linewidth=3, markersize=9)
plt.title('Average Bike Rentals by Month')
plt.xlabel('Month')
plt.ylabel('Avg. Rented Bike Count')
plt.xticks(rotation=45)
plt.show()


**Insight:** **Summer** sees the highest bike demand while **Winter** sees the lowest — an expected pattern since extreme cold discourages cycling. Demand steadily rises from Feb to June and declines from Sept to Jan.

## 11. Impact of Weather Conditions

This addresses the *"weather"* factor from the problem statement — examining how
temperature, humidity, wind, rainfall, snowfall, visibility, and solar radiation
relate to demand.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.scatterplot(data=df_active.sample(2000, random_state=42), x='Temperature', y='Rented_Bike_Count',
                 alpha=0.4, color='#2E86AB', ax=axes[0,0])
axes[0,0].set_title('Temperature vs Rented Bike Count')

sns.scatterplot(data=df_active.sample(2000, random_state=42), x='Humidity', y='Rented_Bike_Count',
                 alpha=0.4, color='#F18F01', ax=axes[0,1])
axes[0,1].set_title('Humidity vs Rented Bike Count')

sns.scatterplot(data=df_active.sample(2000, random_state=42), x='Wind_Speed', y='Rented_Bike_Count',
                 alpha=0.4, color='#6A994E', ax=axes[1,0])
axes[1,0].set_title('Wind Speed vs Rented Bike Count')

sns.scatterplot(data=df_active.sample(2000, random_state=42), x='Visibility', y='Rented_Bike_Count',
                 alpha=0.4, color='#C73E1D', ax=axes[1,1])
axes[1,1].set_title('Visibility vs Rented Bike Count')

plt.tight_layout()
plt.show()


**Insight:** **Temperature** shows the clearest positive relationship with demand — rentals rise with warmer temperatures up to a comfortable range and taper at extremes. Humidity shows a mild negative trend, while wind speed and visibility show weaker relationships.

In [ ]:
# Effect of Rainfall & Snowfall (binary: any precipitation vs none)
df_active['Rain_Flag'] = np.where(df_active['Rainfall'] > 0, 'Rain', 'No Rain')
df_active['Snow_Flag'] = np.where(df_active['Snowfall'] > 0, 'Snow', 'No Snow')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=df_active, x='Rain_Flag', y='Rented_Bike_Count', palette='Blues', ax=axes[0])
axes[0].set_title('Avg. Rentals: Rain vs No Rain')

sns.barplot(data=df_active, x='Snow_Flag', y='Rented_Bike_Count', palette='PuBu', ax=axes[1])
axes[1].set_title('Avg. Rentals: Snow vs No Snow')

plt.tight_layout()
plt.show()


**Insight:** Both **rainfall** and **snowfall** sharply reduce average bike demand, confirming that precipitation is a strong deterrent to cycling.

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_active.sample(2000, random_state=42), x='Solar_Radiation',
                 y='Rented_Bike_Count', alpha=0.4, color='#E9C46A')
plt.title('Solar Radiation vs Rented Bike Count')
plt.xlabel('Solar Radiation (MJ/m2)')
plt.ylabel('Rented Bike Count')
plt.show()


## 12. Holiday & Functioning Day Effects


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=df_active, x='Holiday', y='Rented_Bike_Count', palette='Set2', ax=axes[0])
axes[0].set_title('Avg. Rentals: Holiday vs No Holiday')

sns.barplot(data=df, x='Functioning_Day', y='Rented_Bike_Count', palette='Set1', ax=axes[1])
axes[1].set_title('Avg. Rentals: Functioning Day (System Open) vs Closed')

plt.tight_layout()
plt.show()


**Insight:** Rentals are noticeably **lower on holidays** than regular working days (again pointing to commute-driven demand). As expected, when the system is **non-functioning**, rentals are 0.

## 13. Correlation Heatmap — Numerical Features

A correlation matrix helps identify which numerical weather variables most
strongly influence bike demand — useful groundwork for the predictive model.


In [ ]:
numeric_cols = ['Rented_Bike_Count', 'Hour', 'Temperature', 'Humidity', 'Wind_Speed',
                 'Visibility', 'Dew_Point_Temp', 'Solar_Radiation', 'Rainfall', 'Snowfall']

corr = df_active[numeric_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()


**Insight:** `Temperature` and `Dew_Point_Temp` show the strongest positive correlation with demand, while `Humidity` correlates negatively. `Temperature` and `Dew_Point_Temp` are themselves highly correlated with each other (multicollinearity to watch for in modelling).

## 14. Combined View — Hour × Day of Week Heatmap

A pivot heatmap gives a complete picture of *when* demand is highest across the week.


In [ ]:
pivot = df_active.pivot_table(index='Day_of_Week', columns='Hour',
                               values='Rented_Bike_Count', aggfunc='mean').reindex(day_order)

plt.figure(figsize=(16, 7))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, cbar_kws={'label': 'Avg. Rented Bike Count'})
plt.title('Average Bike Demand: Day of Week × Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.show()


**Insight:** The heatmap clearly visualizes the **dual commute peaks on weekdays** (bright bands around 8 AM and 6 PM) versus the **single midday bulge on weekends**, confirming earlier findings in one consolidated view.

## 15. Interactive Exploration — Season-wise Hourly Pattern (Plotly)


In [ ]:
season_hour = df_active.groupby(['Seasons', 'Hour'])['Rented_Bike_Count'].mean().reset_index()

fig = px.line(season_hour, x='Hour', y='Rented_Bike_Count', color='Seasons',
              title='Hourly Demand Pattern Across Seasons',
              labels={'Rented_Bike_Count': 'Avg. Rented Bike Count'},
              category_orders={'Seasons': season_order},
              color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(template='plotly_white', xaxis=dict(dtick=1))
fig.show()


## 16. Overall Time Series Trend


In [ ]:
daily = df_active.groupby('Date')['Rented_Bike_Count'].sum().reset_index()

fig = px.line(daily, x='Date', y='Rented_Bike_Count',
              title='Total Daily Bike Rentals Over Time',
              labels={'Rented_Bike_Count': 'Total Daily Rentals'})
fig.update_traces(line_color='#2E86AB')
fig.update_layout(template='plotly_white')
fig.show()


**Insight:** The daily trend confirms the seasonal cycle seen earlier — demand rises through spring into summer and declines through autumn into winter, with day-to-day fluctuations likely driven by weather events (rain/snow).

## 17. Key Insights Summary

| # | Insight |
|---|---|
| 1 | Demand peaks at **8 AM and 6 PM** on weekdays — driven by commuters |
| 2 | **Weekends** show a single broader midday peak — more recreational usage |
| 3 | **Summer** has the highest demand; **Winter** the lowest |
| 4 | **Temperature** is the strongest positive weather driver of demand |
| 5 | **Rainfall and Snowfall** sharply reduce rentals |
| 6 | **Humidity** correlates negatively with demand |
| 7 | Demand is **lower on holidays** compared to regular working days |
| 8 | The **Day × Hour heatmap** is the clearest single view of demand patterns |

## 18. Conclusion & Next Steps

This visualization project confirms that bike-sharing demand is strongly shaped
by **time-of-day**, **day-of-week**, and **weather conditions** — exactly the
factors identified in the problem statement. These findings directly inform
feature selection for the next phase: building a **regression / machine learning
model** (e.g., Random Forest, XGBoost, or Linear Regression) to forecast
`Rented_Bike_Count`, using Hour, Temperature, Humidity, Season, Day-of-Week, and
Holiday status as key predictors — enabling better bike allocation and improved
system efficiency.
